**`harmonize_admin`**

Recipe-driven admin geometry harmonization using `Harmonizer`.

Builds only the layers needed to draw one place in its context.

Pass an `admin_id` and the notebook derives the layer set from its
ancestors: the country outline, then each level scoped to the ancestor it
sits inside. `US-MA-SOM` builds the outline of `US`, the level-2 units of
`US`, and the level-3 units of `US-MA`.

Each layer is stored under its parent, so a level-3 file holds one level-2
unit's children rather than the world's. Drawing a single town reads about
400 units instead of 52,000.

Phases, per layer:
1. Discover `stage='ingest'` admin recipes at that level
2. Assign each unit in scope to the highest-priority source (most-specific
   `admin_id` wins, newest version breaks ties)
3. Load and merge geometries across sources
4. Simplify (`simplify_coverage`, so shared borders stay shared)
5. Save

Pass `--admin_levels` instead to build a level globally. That is a
deliberate bulk run, not the default.

See `src/openplaces/recipes/_all/admin/openplaces/2026/` for the recipes.

# Configure

In [ ]:
import argparse

from openplaces.io.admin import context_layers
from openplaces.io.harmonizer import Harmonizer

In [ ]:
parser = argparse.ArgumentParser(description='Admin geometry harmonization')
parser.add_argument(
    '--admin_id',
    help=(
        'Build the layers needed to draw this unit in context, e.g. '
        '"US-MA-SOM". Each layer is scoped to the ancestor it sits inside.'
    ),
    type=str,
    default=None,
)
parser.add_argument(
    '--admin_levels',
    help=(
        'Build these levels globally instead, e.g. "1 2 3 4". A whole-world '
        'build; prefer --admin_id unless you actually want every unit.'
    ),
    nargs='*',
    type=int,
    default=None,
)
parser.add_argument(
    '--reprocess',
    help='Reprocess layers even if output already exists',
    action='store_true',
)
parser.add_argument(
    '--verbose',
    action='store_true',
)

# Test arguments

In [ ]:
ARGS_TEST = '--admin_id US-MA-SOM --reprocess --verbose'

args_list = [x for x in ARGS_TEST.split(' ') if x]
args = parser.parse_args(args_list)
args

# Run

In [ ]:
if args.admin_id and args.admin_levels:
    raise SystemExit('Pass --admin_id or --admin_levels, not both.')
if not args.admin_id and not args.admin_levels:
    raise SystemExit('Pass --admin_id (one place) or --admin_levels (bulk).')

if args.admin_id:
    # Each layer is scoped to the ancestor it sits inside, so only the
    # neighbourhood around the requested unit is built. The scope is the
    # parent whose children the layer holds, which is the level the
    # recipe's `save_to.admin_level` partitions by.
    jobs = [
        (layer['admin_level'], layer['scope'])
        for layer in context_layers(args.admin_id)
    ]
else:
    # No scope: every unit at each level, worldwide.
    jobs = [(admin_level, None) for admin_level in args.admin_levels]

for admin_level, scope in jobs:
    if args.verbose:
        print(f'\nadmin{admin_level} within {scope or "the whole spine"}')
    Harmonizer(
        f'admin-openplaces-2026_admin{admin_level}',
        admin_ids=scope,
        verbose=args.verbose,
    ).harmonize(reprocess=args.reprocess)

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script, test_script

COMMIT = True
# If True, writes `.py` scripts to 'scripts/'.
# If False, writes a test version of the script to 'scripts/_test/...'

In [ ]:
convert_to_script(commit=COMMIT)

# Test script

In [ ]:
test_script(*args_list, committed=COMMIT)

# Inspect output

In [ ]:
from openplaces.api import get_admin

# The innermost layer built: the one holding the requested unit among its
# siblings. A country has no such layer -- it *is* the outline -- so fall
# back to that.
if args.admin_id:
    layers = context_layers(args.admin_id)
    focus = next(
        (layer for layer in reversed(layers) if layer['role'] == 'focus'),
        layers[0],
    )
    admin_level, scope = focus['admin_level'], focus['scope']
else:
    admin_level, scope = args.admin_levels[0], None

admin = get_admin(admin_id=scope, level=admin_level, geom=True)
print(f'{len(admin):,d} admin{admin_level} units within {scope or "the world"}')
admin.head()